# XAI Short Smoke Run

Notebook nay chay mot run XAI ngan 3 epoch de xac nhan artifact moi sau khi sua trainer.
Muc tieu chinh la kiem tra `train_saliency_loss`, `train_energy_in_box`, `train_pointing_game`, va `train_saliency_iou` co con bi dung im qua cac epoch hay khong.


In [ ]:
!pip install ultralytics==8.4.61

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Thanhmay2406/xai-driven-saliency-loss.git"
KAGGLE_WORKING = Path("/kaggle/working")
DEFAULT_REPO_ROOT = KAGGLE_WORKING / "xai-driven-saliency-loss"

if DEFAULT_REPO_ROOT.exists():
    REPO_ROOT = DEFAULT_REPO_ROOT
    %cd {REPO_ROOT}
    !git pull --ff-only
else:
    REPO_ROOT = DEFAULT_REPO_ROOT
    !git clone {REPO_URL} {REPO_ROOT}
    %cd {REPO_ROOT}

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

In [ ]:
import json
import random

import numpy as np
import pandas as pd
import torch

from src.training import UltralyticsYOLOXAITrainerConfig, train_ultralytics_yolo_xai

CFG = {
    "DATASET_YAML": "/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml",
    "INIT_WEIGHTS": "yolo11n.pt",
    "PROJECT_DIR": "/kaggle/working/yolo_xai_short_smoke",
    "XAI_RUN_NAME": "xai_from_scratch_v2_smoke_fix",
    "IMGSZ": 640,
    "BATCH": 16,
    "DEVICE": 0,
    "WORKERS": 4,
    "EPOCHS": 3,
    "PATIENCE": 3,
    "SEED": 42,
    "OPTIMIZER": "SGD",
    "LR": 1e-2,
    "LRF": 1e-2,
    "MOMENTUM": 0.937,
    "WEIGHT_DECAY": 5e-4,
    "WARMUP_EPOCHS": 3,
    "XAI_METHOD": "activation",
    "LAMBDA_SALIENCY": 0.006,
    "NUM_CLASSES": 6,
    "USE_PROGRESSIVE_LAMBDA": True,
    "PROGRESSIVE_WARMUP_EPOCHS": 30,
    "GT_MASK_MODE": "soft",
    "SOFT_MASK_SIGMA": 0.35,
    "SHRUNK_BOX_RATIO": 0.7,
}

DATASET_YAML = Path(CFG["DATASET_YAML"])
if not DATASET_YAML.exists():
    raise FileNotFoundError(f"Khong tim thay dataset yaml: {DATASET_YAML}")

Path(CFG["PROJECT_DIR"]).mkdir(parents=True, exist_ok=True)

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])

xai_cfg = UltralyticsYOLOXAITrainerConfig(
    xai_method=CFG["XAI_METHOD"],
    lambda_saliency=CFG["LAMBDA_SALIENCY"],
    num_classes=CFG["NUM_CLASSES"],
    use_progressive_lambda=CFG["USE_PROGRESSIVE_LAMBDA"],
    progressive_warmup_epochs=CFG["PROGRESSIVE_WARMUP_EPOCHS"],
    gt_mask_mode=CFG["GT_MASK_MODE"],
    soft_mask_sigma=CFG["SOFT_MASK_SIGMA"],
    shrunk_box_ratio=CFG["SHRUNK_BOX_RATIO"],
)

train_overrides = {
    "data": str(DATASET_YAML),
    "project": CFG["PROJECT_DIR"],
    "name": CFG["XAI_RUN_NAME"],
    "model": str(CFG["INIT_WEIGHTS"]),
    "epochs": CFG["EPOCHS"],
    "imgsz": CFG["IMGSZ"],
    "batch": CFG["BATCH"],
    "device": CFG["DEVICE"],
    "workers": CFG["WORKERS"],
    "patience": CFG["PATIENCE"],
    "seed": CFG["SEED"],
    "optimizer": CFG["OPTIMIZER"],
    "lr0": CFG["LR"],
    "lrf": CFG["LRF"],
    "momentum": CFG["MOMENTUM"],
    "weight_decay": CFG["WEIGHT_DECAY"],
    "warmup_epochs": CFG["WARMUP_EPOCHS"],
    "plots": False,
    "val": True,
}

print(json.dumps(CFG, indent=2))

In [ ]:
trainer = train_ultralytics_yolo_xai(
    train_overrides=train_overrides,
    xai_config=xai_cfg,
)

run_dir = Path(trainer.save_dir)
history_json_path = run_dir / "weights" / "train_history.json"
history_csv_path = run_dir / "weights" / "train_history.csv"
summary_path = run_dir / "summary" / "run_summary.json"

print("Run dir:", run_dir)
print("History CSV:", history_csv_path)
print("Summary:", summary_path)

history_df = pd.read_csv(history_csv_path)
display(history_df)

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
metric_cols = [
    "train_saliency_loss",
    "train_energy_in_box",
    "train_pointing_game",
    "train_saliency_iou",
]

diagnostics = []
for col in metric_cols:
    values = history_df[col].tolist()
    unique_count = int(history_df[col].nunique(dropna=False))
    diagnostics.append({
        "metric": col,
        "unique_count": unique_count,
        "first": values[0],
        "last": values[-1],
        "values": values,
    })

diag_df = pd.DataFrame(diagnostics)
display(diag_df)

frozen_metrics = [item["metric"] for item in diagnostics if item["unique_count"] <= 1]
if frozen_metrics:
    raise AssertionError(f"Cac metric van bi dung im: {frozen_metrics}")

if float(history_df["train_saliency_loss"].sum()) <= 0.0:
    raise AssertionError("train_saliency_loss van bang 0 tren toan bo smoke run")

print("Smoke run passed: XAI metrics da thay doi qua cac epoch.")